## Connecting Python to SQL Server

In this section, we establish a connection between Python and our SQL Server database using SQLAlchemy and PyODBC.

The database used for this project is `HotelIQ`.

In [ ]:
import pandas as pd
import pyodbc
import sqlalchemy
from sqlalchemy import create_engine
from urllib.parse import quote_plus

In [ ]:



# SQL Server connection details
connection_string = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=YUKTA\\SQLEXPRESS02;"
    "DATABASE=HotelIQ;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

# Create SQLAlchemy engine
connection_url = quote_plus(connection_string)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={connection_url}",
    fast_executemany=True
)



Successfully connected to HotelIQ database!


In [ ]:
# Test database connection
try:
    with engine.connect() as connection:
        print("Successfully connected to HotelIQ database!")

except Exception as e:
    print("Connection failed:")
    print(e)

In [6]:
# load cleaned dataset

df = pd.read_csv('../data/hotel_bookings_cleaned.csv')
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,booking_changes,deposit_type,agent,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,3,No Deposit,-1.0,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,4,No Deposit,-1.0,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,0,No Deposit,-1.0,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,0,No Deposit,304.0,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,0,No Deposit,240.0,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [5]:
print('Dataset Shape: ',df.shape)

Dataset Shape:  (87395, 31)


In [7]:
df.to_sql(name='hotel_bookings',
          con=engine,
          if_exists='replace',
          index=False,
          chunksize=1000)

print("Dataset successfully uploaded to DQL server!")

Dataset successfully uploaded to DQL server!


In [8]:
# Verifying uploaded data
query = """
SELECT COUNT(*) AS Total_Records
FROM hotel_bookings;
"""

result = pd.read_sql(query,engine)

result

,Total_Records
0,87395


## SQL Business Analysis

#### Business Question 1: Which hotel type receives more bookings?

**Objective:** Identify which hotel type, City Hotel or Resort Hotel, receives the highest number of bookings.

In [9]:
query = """
SELECT hotel, COUNT(*) AS total_bookings
FROM hotel_bookings
GROUP BY hotel
ORDER BY total_bookings DESC;
"""

result = pd.read_sql(query, engine)

result

,hotel,total_bookings
0,City Hotel,53428
1,Resort Hotel,33967


#### Interpretation

- City Hotel received 53,428 bookings, accounting for approximately 61.13% of total bookings.

- Resort Hotel received 33,967 bookings (38.87%), indicating that City Hotel had a higher booking volume.



#### Business Question 2: What is the cancellation rate for each hotel type?

**Objective:** Compare cancellation rates between City Hotels and Resort Hotels to identify which hotel type experiences a higher proportion of booking cancellations.

In [11]:
query = """
SELECT hotel, COUNT(*) AS total_bookings,
SUM(is_canceled) AS canceled_bookings,
ROUND(SUM(is_canceled)*100.0/COUNT(*), 2) AS cancellation_rate
FROM hotel_bookings
GROUP BY hotel
ORDER BY cancellation_rate DESC;
"""

result = pd.read_sql(query, engine)

result

,hotel,total_bookings,canceled_bookings,cancellation_rate
0,City Hotel,53428,16049,30.04
1,Resort Hotel,33967,7976,23.48


#### Interpretation

- City Hotel recorded a cancellation rate of 30.04%, compared to 23.48% for Resort Hotel.

- This shows that cancellations were more frequent at City Hotels, with a difference of 6.56 percentage points.



#### Business Question 3: Which market segment generates the most bookings?

**Objective:** Identify the market segments contributing the highest number of hotel bookings and understand their contribution to overall booking volume.

In [12]:
query = """
SELECT
    market_segment,
    COUNT(*) AS total_bookings
FROM hotel_bookings
GROUP BY market_segment
ORDER BY total_bookings DESC;
"""

result = pd.read_sql(query, engine)

result

,market_segment,total_bookings
0,Online TA,51618
1,Offline TA/TO,13889
2,Direct,11804
3,Groups,4941
4,Corporate,4212
5,Complementary,702
6,Aviation,227
7,Undefined,2


#### Interpretation

- Online TA was the leading market segment, with 51,618 bookings, contributing approximately 59.06% of total bookings.

- Offline TA/TO ranked second with 13,889 bookings, followed by Direct bookings with 11,804.

#### Business Question 4: Which months have the highest number of bookings?

**Objective:** Identify the months with the highest and lowest booking volumes to understand seasonal booking patterns.

In [13]:
query = """
SELECT
    arrival_date_month,
    COUNT(*) AS total_bookings
FROM hotel_bookings
GROUP BY arrival_date_month
ORDER BY
    CASE arrival_date_month
        WHEN 'January' THEN 1
        WHEN 'February' THEN 2
        WHEN 'March' THEN 3
        WHEN 'April' THEN 4
        WHEN 'May' THEN 5
        WHEN 'June' THEN 6
        WHEN 'July' THEN 7
        WHEN 'August' THEN 8
        WHEN 'September' THEN 9
        WHEN 'October' THEN 10
        WHEN 'November' THEN 11
        WHEN 'December' THEN 12
    END;
"""

result = pd.read_sql(query, engine)

result

,arrival_date_month,total_bookings
0,January,4693
1,February,6098
2,March,7512
3,April,7908
4,May,8355
5,June,7765
6,July,10057
7,August,11257
8,September,6690
9,October,6934


#### Interpretation

- August recorded the highest number of bookings (11,257), followed by July (10,057).

- January had the lowest booking volume, with 4,693 bookings.

- These figures represent combined bookings across all three years in the dataset (2015–2017).

#### Business Question 5: Which customer type has the highest cancellation rate?

**Objective:** Compare booking cancellation rates across different customer types to identify which category has the highest proportion of cancellations.

In [14]:
query = """
SELECT
    customer_type,
    COUNT(*) AS total_bookings,
    SUM(is_canceled) AS canceled_bookings,
    ROUND(
        SUM(is_canceled) * 100.0 / COUNT(*), 2
    ) AS cancellation_rate
FROM hotel_bookings
GROUP BY customer_type
ORDER BY cancellation_rate DESC;
"""

result = pd.read_sql(query, engine)

result

,customer_type,total_bookings,canceled_bookings,cancellation_rate
0,Transient,71986,21672,30.11
1,Contract,3139,512,16.31
2,Transient-Party,11726,1787,15.24
3,Group,544,54,9.93


#### Interpretation

- Transient customers recorded the highest cancellation rate of 30.11%, with 21,672 canceled bookings out of 71,986 bookings.

- Contract and Transient-Party customers had cancellation rates of 16.31% and 15.24%, respectively.

- Group customers recorded the lowest cancellation rate at 9.93%, indicating comparatively fewer cancellations in this category.

#### Business Question 6: What is the average ADR for each hotel type?

**Objective:** Compare the average daily rate (ADR) between City Hotel and Resort Hotel to understand their pricing differences.

In [15]:
query = """
SELECT
    hotel,
    ROUND(AVG(adr), 2) AS average_adr
FROM hotel_bookings
GROUP BY hotel
ORDER BY average_adr DESC;
"""

result = pd.read_sql(query, engine)

result

,hotel,average_adr
0,City Hotel,110.99
1,Resort Hotel,99.03


#### Interpretation

- City Hotel recorded an average ADR of 110.99, which was higher than Resort Hotel's average ADR of 99.03.

- This indicates that City Hotels had a higher average daily room rate than Resort Hotels during the period covered by the dataset.

#### Business Question 7: Which months have the highest cancellation rates?

**Objective:** Identify the months with the highest proportion of canceled bookings.

In [16]:
query = """
SELECT
    arrival_date_month,
    COUNT(*) AS total_bookings,
    SUM(is_canceled) AS canceled_bookings,
    ROUND(
        SUM(is_canceled) * 100.0 / COUNT(*), 2
    ) AS cancellation_rate
FROM hotel_bookings
GROUP BY arrival_date_month
ORDER BY cancellation_rate DESC;
"""

result = pd.read_sql(query, engine)
result

,arrival_date_month,total_bookings,canceled_bookings,cancellation_rate
0,August,11257,3623,32.18
1,July,10057,3198,31.80
2,April,7908,2409,30.46
3,June,7765,2354,30.32
4,May,8355,2442,29.23
5,December,5131,1378,26.86
6,September,6690,1642,24.54
7,March,7512,1830,24.36
8,October,6934,1642,23.68
9,February,6098,1415,23.20


#### Interpretation

- August recorded the highest cancellation rate at 32.18%, followed by July at 31.80%.

- November had the lowest cancellation rate at 21.10%, followed by January at 22.12%.

- The results show variations in cancellation rates across different arrival months.

#### Business Question 8: How does lead time affect booking cancellations?

**Objective:** Compare cancellation rates across different booking lead-time groups.

In [23]:
query = """
SELECT
    CASE
        WHEN lead_time <= 30 THEN '0-30 Days'
        WHEN lead_time <= 90 THEN '31-90 Days'
        WHEN lead_time <= 180 THEN '91-180 Days'
        WHEN lead_time <= 365 THEN '181-365 Days'
        ELSE 'Above 365 Days'
    END AS lead_time_group,

    COUNT(*) AS total_bookings,

    SUM(is_canceled) AS canceled_bookings,

    ROUND(
        SUM(is_canceled) * 100.0 / COUNT(*), 2
    ) AS cancellation_rate

FROM hotel_bookings

GROUP BY
    CASE
        WHEN lead_time <= 30 THEN '0-30 Days'
        WHEN lead_time <= 90 THEN '31-90 Days'
        WHEN lead_time <= 180 THEN '91-180 Days'
        WHEN lead_time <= 365 THEN '181-365 Days'
        ELSE 'Above 365 Days'
    END

ORDER BY
    MIN(lead_time);
"""

result = pd.read_sql(query, engine)

result

,lead_time_group,total_bookings,canceled_bookings,cancellation_rate
0,0-30 Days,34644,5688,16.42
1,31-90 Days,22744,7280,32.01
2,91-180 Days,18243,6382,34.98
3,181-365 Days,11199,4444,39.68
4,Above 365 Days,565,231,40.88


#### Interpretation

- Bookings made 0–30 days before arrival had the lowest cancellation rate at 16.42%.

- Cancellation rates generally increased with longer lead times, reaching 39.68% for bookings made 181–365 days in advance.

- Bookings made more than 365 days in advance recorded the highest cancellation rate at 40.88%, although this group contained only 565 bookings.

#### Business Question 9: Which market segments have the highest cancellation rates?

**Objective:** Analyse cancellation rates across different market segments.

In [19]:
query = """
SELECT
    market_segment,
    COUNT(*) AS total_bookings,
    SUM(is_canceled) AS canceled_bookings,
    ROUND(
        SUM(is_canceled) * 100.0 / COUNT(*), 2
    ) AS cancellation_rate
FROM hotel_bookings
GROUP BY market_segment
ORDER BY cancellation_rate DESC;
"""

result = pd.read_sql(query, engine)
result

,market_segment,total_bookings,canceled_bookings,cancellation_rate
0,Undefined,2,2,100.00
1,Online TA,51618,18245,35.35
2,Groups,4941,1335,27.02
3,Aviation,227,45,19.82
4,Offline TA/TO,13889,2063,14.85
5,Direct,11804,1737,14.72
6,Complementary,702,88,12.54
7,Corporate,4212,510,12.11


#### Interpretation

- Online TA recorded the highest cancellation rate among the major market segments at 35.35%, with 18,245 canceled bookings.

- Groups had a cancellation rate of 27.02%, while Direct and Offline TA/TO recorded considerably lower rates of 14.72% and 14.85%, respectively.

- Undefined had a 100% cancellation rate, but this was based on only 2 bookings and should not be considered representative.

#### Business Question 10: Which distribution channel generates the most bookings?

**Objective:** Compare booking volumes across different distribution channels.

In [20]:
query = """
SELECT
    distribution_channel,
    COUNT(*) AS total_bookings
FROM hotel_bookings
GROUP BY distribution_channel
ORDER BY total_bookings DESC;
"""

result = pd.read_sql(query, engine)
result

,distribution_channel,total_bookings
0,TA/TO,69141
1,Direct,12987
2,Corporate,5081
3,GDS,181
4,Undefined,5


#### Interpretation

- TA/TO was the leading distribution channel, generating 69,141 bookings, making it the dominant channel in the dataset.

- Direct bookings ranked second with 12,987 bookings, followed by Corporate with 5,081 bookings.

- GDS and Undefined contributed very few bookings compared to the other distribution channels.

#### Business Question 11: What is the relationship between special requests and booking cancellations?

**Objective:** Examine whether cancellation rates differ based on the number of special requests made by customers.

In [21]:
query = """
SELECT
    total_of_special_requests,
    COUNT(*) AS total_bookings,
    SUM(is_canceled) AS canceled_bookings,
    ROUND(
        SUM(is_canceled) * 100.0 / COUNT(*), 2
    ) AS cancellation_rate
FROM hotel_bookings
GROUP BY total_of_special_requests
ORDER BY total_of_special_requests;
"""

result = pd.read_sql(query, engine)
result

,total_of_special_requests,total_bookings,canceled_bookings,cancellation_rate
0,0,43893,14574,33.20
1,1,29017,6503,22.41
2,2,11812,2515,21.29
3,3,2317,397,17.13
4,4,320,34,10.63
5,5,36,2,5.56


#### Interpretation

- Bookings with zero special requests recorded the highest cancellation rate at 33.20%, with 14,574 cancellations.

- Cancellation rates generally decreased as the number of special requests increased, reaching 10.63% for four requests.

- Bookings with five special requests had the lowest cancellation rate at 5.56%, although this category contained only 36 bookings.

#### Business Question 12: Which countries contribute the most bookings?

**Objective:** Identify the countries with the highest number of hotel bookings.

In [22]:
query = """
SELECT TOP 10
    country,
    COUNT(*) AS total_bookings
FROM hotel_bookings
GROUP BY country
ORDER BY total_bookings DESC;
"""

result = pd.read_sql(query, engine)
result

,country,total_bookings
0,PRT,27453
1,GBR,10432
2,FRA,8837
3,ESP,7252
4,DEU,5387
5,ITA,3066
6,IRL,3016
7,BEL,2081
8,BRA,1995
9,NLD,1911


#### Interpretation

- Portugal (PRT) contributed the highest number of bookings, with 27,453, followed by the United Kingdom (GBR) with 10,432 bookings.

- France (FRA) ranked third with 8,837 bookings, followed by Spain (ESP) with 7,252 bookings.

- The top five countries were Portugal, United Kingdom, France, Spain, and Germany, indicating that these countries contributed substantially to the bookings in the dataset.

## Overall SQL Analysis Summary

The SQL analysis provided useful insights into hotel booking patterns, customer behaviour, pricing, and cancellation trends.

### Key Findings

1. **Hotel Booking Distribution:** City Hotel received more bookings (53,428) than Resort Hotel (33,967), accounting for approximately 61.13% of total bookings.

2. **Cancellation Patterns:** City Hotel recorded a higher cancellation rate (30.04%) compared to Resort Hotel (23.48%). Transient customers had the highest cancellation rate among customer types at 30.11%.

3. **Market and Distribution Channels:** Online TA was the largest market segment, contributing 51,618 bookings. Similarly, TA/TO was the leading distribution channel with 69,141 bookings.

4. **Seasonal Trends:** August recorded the highest booking volume (11,257) and the highest cancellation rate (32.18%), while January had the lowest booking volume (4,693). November recorded the lowest cancellation rate (21.10%).

5. **Pricing Analysis:** City Hotel had a higher average ADR of 110.99 compared to Resort Hotel's 99.03.

6. **Lead Time Analysis:** Cancellation rates generally increased with longer booking lead times, rising from 16.42% for bookings made within 30 days to 40.88% for bookings made more than 365 days in advance.

7. **Special Requests:** Bookings without special requests had the highest cancellation rate (33.20%), while bookings with more special requests generally showed lower cancellation rates.

8. **Country-wise Bookings:** Portugal contributed the highest number of bookings (27,453), followed by the United Kingdom (10,432) and France (8,837).

### Conclusion

Overall, the SQL analysis helped identify important patterns in booking distribution, cancellation behaviour, customer types, pricing, and distribution channels. City Hotel dominated booking volume, while longer lead times and certain customer and market segments were associated with higher cancellation rates.

These findings provide a foundation for further statistical analysis and machine learning, particularly for understanding cancellation patterns and developing a hotel booking cancellation prediction model.